# ViDoRe Finance–EN

**Question:** Does representing the same bank 10-K page as OCR text or as an image change which evidence we find?

The saved results cover the entire Finance–EN corpus and all six query languages. This notebook reads saved outputs and runs without an API key or GPU.

In [1]:
import csv
import json
from html import escape
from pathlib import Path
from IPython.display import HTML

results = Path("results")

def read_csv(name):
    with (results / name).open(encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))

def read_json(name):
    return json.loads((results / name).read_text(encoding="utf-8"))

def show(columns, rows):
    head = "".join(f"<th>{escape(str(cell))}</th>" for cell in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(cell))}</td>" for cell in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def score(value):
    return f"{float(value):.4f}" if value != "" else "n/a"

labels = {
    "bm25_ocr": "BM25 OCR", "dense_ocr": "Qwen OCR",
    "dense_image": "Qwen image", "rrf_ocr_image": "Text + image RRF",
}
manifest = read_json("run_manifest.json")
summary = read_csv("summary.csv")
show(["Source reports", "Pages", "Query variants", "Relevance judgments", "Empty OCR pages"], [
    [manifest["documents"], manifest["pages"], manifest["queries"], manifest["qrels"],
     len(manifest["empty_ocr_pages"])]
])
show(["Language", "Questions"], sorted(manifest["languages"].items()))

Source reports,Pages,Query variants,Relevance judgments,Empty OCR pages
6,2942,1854,8766,19


Language,Questions
english,309
french,309
german,309
italian,309
portuguese,309
spanish,309


## Methods

- **BM25 OCR** ranks pages by the extracted words in the query.
- **Qwen OCR** compares the question embedding with embeddings of OCR page text.
- **Qwen image** uses the same Qwen checkpoint to embed page images. Both dense runs share the same question vector.
- **Text + image RRF** combines the two dense top-100 lists with equal weights and rank constant 60.

All methods search the same page pool. Dense search scores every page by exact cosine similarity. The page representations are cached and neither labels nor reference answers enter the retrieval query.

## English retrieval results

Hit checks for any relevant page. Recall counts what fraction of the labeled pages is retrieved. Graded nDCG gives greater credit to fully relevant pages and to earlier ranks. Complete-answer-page Hit@5 is evaluated only where the publisher marked at least one page as containing a complete answer.

In [2]:
english = {(r["method"], r["k"]): r for r in summary if r["scope"] == "language" and r["group"] == "english"}
show(["Method", "Hit@5", "Recall@5", "nDCG@10", "Grade-2 hit@5", "Eligible questions"], [
    [label, score(english[method, "5"]["hit"]), score(english[method, "5"]["recall"]),
     score(english[method, "10"]["ndcg"]), score(english[method, "5"]["complete_page_hit"]),
     english[method, "5"]["complete_page_eligible_queries"]]
    for method, label in labels.items()
])
paired = read_csv("paired_english.csv")
show(["Baseline", "Candidate", "Metric", "Mean change", "Wins", "Losses", "Ties"], [
    [r["baseline"], r["candidate"], r["metric"], score(r["mean_delta"]), r["wins"], r["losses"], r["ties"]]
    for r in paired
])

Method,Hit@5,Recall@5,nDCG@10,Grade-2 hit@5,Eligible questions
BM25 OCR,0.7896,0.4727,0.5302,0.7523,222
Qwen OCR,0.7929,0.4715,0.5275,0.7207,222
Qwen image,0.8673,0.5373,0.5931,0.8153,222
Text + image RRF,0.8544,0.5146,0.5760,0.7883,222


Baseline,Candidate,Metric,Mean change,Wins,Losses,Ties
Qwen OCR,Qwen image,hit@5,0.0744,36,13,260
Qwen OCR,Qwen image,recall@5,0.0658,82,52,175
Qwen OCR,Qwen image,ndcg@10,0.0656,152,87,70
Qwen OCR,Text + image RRF,hit@5,0.0615,23,4,282
Qwen OCR,Text + image RRF,recall@5,0.0431,57,25,227
Qwen OCR,Text + image RRF,ndcg@10,0.0485,138,75,96
Qwen image,Text + image RRF,hit@5,-0.0129,11,15,283
Qwen image,Text + image RRF,recall@5,-0.0227,40,46,223
Qwen image,Text + image RRF,ndcg@10,-0.0172,106,110,93


## Language and content breakdown

Each language column has 309 questions. Translations share the underlying tasks, so they are not independent extra questions. The English content groups use publisher tags and prioritize chart, image or infographic tags above table tags.

In [3]:
def grouped_ndcg(scope):
    rows = [r for r in summary if r["scope"] == scope and r["k"] == "10"]
    groups = sorted({r["group"] for r in rows})
    by_key = {(r["group"], r["method"]): r for r in rows}
    show(["Group", "Questions", *labels.values()], [
        [group, by_key[group, "bm25_ocr"]["queries"],
         *(score(by_key[group, method]["ndcg"]) for method in labels)]
        for group in groups
    ])

grouped_ndcg("all_variants")
grouped_ndcg("language")
grouped_ndcg("english_content")

Group,Questions,BM25 OCR,Qwen OCR,Qwen image,Text + image RRF
all,1854,0.1831,0.4631,0.5325,0.5211


Group,Questions,BM25 OCR,Qwen OCR,Qwen image,Text + image RRF
english,309,0.5302,0.5275,0.5931,0.5760
french,309,0.0774,0.4514,0.5148,0.5121
german,309,0.1936,0.4364,0.5081,0.4937
italian,309,0.1605,0.4474,0.5190,0.5149
portuguese,309,0.0915,0.4459,0.5179,0.5023
spanish,309,0.0457,0.4698,0.5417,0.5277


Group,Questions,BM25 OCR,Qwen OCR,Qwen image,Text + image RRF
chart_or_other_visual,23,0.4951,0.6157,0.6661,0.6512
table,155,0.4402,0.4785,0.5415,0.5275
text_or_other,131,0.6427,0.5699,0.6414,0.6201


## Evidence coverage walkthrough

These worked examples mark where specific facts appear. A fact is counted when its mapped page is in the ranking. A relevant page can support only part of the answer. OCR offsets and the official query/page bounding boxes are saved for inspection. The two examples illustrate the method and are distinct from full benchmark scores.

In [4]:
for example in read_json("evidence_examples.json")["examples"]:
    display(HTML(f"<p><b>Query {example['query_id']}</b> — {escape(example['question'])}</p>"))
    show(["Fact", "Page ID", "OCR character interval"], [
        [f["fact"], f["page_id"], f"[{f['ocr_start']}, {f['ocr_end_exclusive']})"]
        for f in example["facts"]
    ])
    show(["Method", "Cutoff", "Page hit", "Facts whose mapped page was retrieved"], [
        [labels[r["method"]], r["k"], r["page_hit"], f"{r['facts_with_located_page']}/{r['required_facts']}"]
        for r in example["retrieval"] if r["k"] in (1, 5)
    ])

Query 45 — Citigroup Services and Markets net income 2023

Fact,Page ID,OCR character interval
"Services 2023 net income was USD 4,635 million.",1141,"[4949, 4954)"
"Markets 2023 net income was USD 3,871 million.",1175,"[4813, 4818)"


Method,Cutoff,Page hit,Facts whose mapped page was retrieved
BM25 OCR,1,True,0/2
BM25 OCR,5,True,0/2
Qwen OCR,1,True,0/2
Qwen OCR,5,True,1/2
Qwen image,1,True,0/2
Qwen image,5,True,2/2
Text + image RRF,1,True,0/2
Text + image RRF,5,True,1/2


Query 111 — List all the subsidiaries registered in the Bahamas as per Exhibit 21.01 Citigroup’s 2024 subsidiary list.

Fact,Page ID,OCR character interval
Citi Investments Bahamas Ltd.,1982,"[2030, 2059)"
Citi Overseas Holdings Bahamas Limited,1982,"[2078, 2116)"
Citi Overseas Investments Bahamas Inc.,1982,"[2683, 2721)"
Citigroup Global Markets Holdings Bahamas Limited,1983,"[533, 582)"


Method,Cutoff,Page hit,Facts whose mapped page was retrieved
BM25 OCR,1,True,3/4
BM25 OCR,5,True,3/4
Qwen OCR,1,True,3/4
Qwen OCR,5,True,4/4
Qwen image,1,True,3/4
Qwen image,5,True,4/4
Text + image RRF,1,True,3/4
Text + image RRF,5,True,4/4


## Where the two page representations rank differently

These are the largest observed English nDCG@10 changes between the two dense runs. Use them for error inspection alongside the paired summary.

In [5]:
shifts = read_json("evidence_examples.json")["rank_shifts"]
show(["Query", "OCR nDCG@10", "Image nDCG@10", "OCR top 5 page IDs", "Image top 5 page IDs"], [
    [r["question"], score(r["ocr_ndcg_at_10"]), score(r["image_ndcg_at_10"]),
     r["ocr_top_five"], r["image_top_five"]]
    for r in shifts
])

Query,OCR nDCG@10,Image nDCG@10,OCR top 5 page IDs,Image top 5 page IDs
By how much were Citigroup's Other Real Estate Owner assets deducted from 2020 to 2024?,1.0000,0.0000,"[1985, 1718, 1101, 1108, 38]","[543, 1929, 557, 1718, 2020]"
Did digital adoption increase for Bank of America in 2024?,0.0000,1.0000,"[2719, 413, 2721, 2638, 417]","[2875, 2849, 413, 2874, 2336]"


## Cost and verification

Index build cost is measured separately from query latency. Query totals combine GPU encoding and CPU ranking; fusion uses a single query encoding. Timing is specific to the recorded laptop run.

In [6]:
timing = [r for r in read_csv("timing_summary.csv") if r["scope"] == "english"]
show(["Method", "Median encoder ms", "Median rank ms", "Estimated median total ms", "Estimated p95 ms"], [
    [labels[r["method"]], *(score(r[field]) for field in ("median_encode_ms", "median_rank_ms",
       "median_estimated_total_ms", "p95_estimated_total_ms"))]
    for r in timing
])
show(["Modality", "Pages encoded", "Encoding seconds", "Matrix assembly seconds", "Vector MiB"], [
    [key, e["rows"], f"{e['encode_seconds']:.1f}", f"{e['matrix_seconds']:.2f}", f"{e['matrix_bytes']/2**20:.1f}"]
    for key, e in manifest["embeddings"].items() if key in ("ocr", "image")
])
print(json.dumps(read_json("audit.json"), indent=2))

Method,Median encoder ms,Median rank ms,Estimated median total ms,Estimated p95 ms
BM25 OCR,0.0000,0.4530,0.4530,0.6310
Qwen OCR,47.6445,0.8824,48.5706,88.6225
Qwen image,47.6445,0.8683,48.5866,88.4748
Text + image RRF,47.6445,1.8948,49.6544,89.4808


Modality,Pages encoded,Encoding seconds,Matrix assembly seconds,Vector MiB
ocr,2942,733.5,0.93,23.0
image,2942,1226.9,1.47,23.0


{
  "passed": true,
  "pages": 2942,
  "query_variants": 1854,
  "metric_rows_checked": 22248,
  "summary_rows_checked": 120,
  "timing_rows_checked": 7416,
  "checks": "Coverage, ranking IDs, qrels, per-query values, grouped means and timing arithmetic."
}


See [FEASIBILITY.md](FEASIBILITY.md) for the recommendation, limits, and follow-up design. The benchmark evaluates retrieval. A shared answer reader could be added in a separate end-to-end experiment.